# Multi-Agent SysML Sequence Diagram Generator
LangGraph + Groq (LLaMA 3.1)


In [ ]:
!pip install langgraph langchain langchain-groq pypdf plantuml

In [ ]:
import os
from langchain_groq import ChatGroq
os.environ['GROQ_API_KEY'] = 'YOUR_API_KEY_HERE'
llm = ChatGroq(model='llama3-8b-8192', temperature=0.2)

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
file_name

In [ ]:
from pypdf import PdfReader
def load_requirements(file_path):
    if file_path.endswith('.txt'):
        return open(file_path).read()
    elif file_path.endswith('.pdf'):
        reader = PdfReader(file_path)
        return '\n'.join([p.extract_text() for p in reader.pages])
    else:
        raise ValueError('Unsupported file type')
requirements_text = load_requirements(file_name)
requirements_text[:500]

In [ ]:
def extract_functionalities(text):
    return llm.invoke(f'Extract functionalities:\n{text}').content

def extract_interactions(func):
    return llm.invoke(f'Extract interactions:\n{func}').content

def generate_diagram(interaction):
    return llm.invoke(f'Generate PlantUML sequence diagram:\n{interaction}').content


In [ ]:
from langgraph.graph import StateGraph
class State(dict): pass

def node1(state):
    state['funcs'] = extract_functionalities(state['req'])
    return state

def node2(state):
    interactions = []
    for f in state['funcs'].split('\n'):
        if f.strip():
            interactions.append(extract_interactions(f))
    state['interactions'] = interactions
    return state

def node3(state):
    diagrams = []
    for i in state['interactions']:
        diagrams.append(generate_diagram(i))
    state['diagrams'] = diagrams
    return state

builder = StateGraph(State)
builder.add_node('n1', node1)
builder.add_node('n2', node2)
builder.add_node('n3', node3)
builder.set_entry_point('n1')
builder.add_edge('n1','n2')
builder.add_edge('n2','n3')
graph = builder.compile()

In [ ]:
result = graph.invoke({'req': requirements_text})
for i,d in enumerate(result['diagrams']):
    print(f'--- Diagram {i+1} ---')
    print(d)